# Imports

In [1]:
%%capture --no-display
!pip install gradio
!pip install -qU langchain-openai
!pip install python-dotenv

In [2]:
import gradio as gr
import numpy as np
import smtplib
import getpass
from langchain_openai import ChatOpenAI
import os
import pandas as pd
import re
from email.mime.image import MIMEImage
from dotenv import load_dotenv
import requests
from langchain_openai import AzureChatOpenAI
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from langchain.prompts.chat import ChatPromptTemplate
from PIL import Image
from langchain.schema.output_parser import OutputParserException
import time
import json
from langchain.schema import (
    SystemMessage,
    HumanMessage,
    AIMessage
)

# CAR CLASSIFIER

In [3]:
class CarTypePredictor:
    def __init__(self):
        self.car_types = ["SUV", "Sedan", "Truck", "Heavy Truck"]

    def predict(self, image):
        return np.random.choice(self.car_types)

# CAR DATA PROCESSOR

In [4]:
class CarDataProcessor:
    def __init__(self):
        self.car_description = ""
        self.car_image = None
        self.sender_email = ""
        self.sender_pass = ""
        self.reciever_email = ""
        self.image_path = "/content/saved_image.jpg"
        self.predictor = CarTypePredictor()

    def process_inputs(self, car_description, sender_email, sender_pass, reciever_email, car_image):
        self.car_description = car_description
        self.car_image = car_image
        self.sender_email = sender_email
        self.sender_pass = sender_pass
        self.reciever_email = reciever_email

        self.tmp_img = Image.fromarray(car_image)
        self.tmp_img.save(self.image_path)


        return self.predictor.predict(car_image)

# AZURE USING LANGCHAIN

In [5]:
class CarInformationExtractor:
    def __init__(self, openai_api_key=None, model_name="gpt-4o-mini", azure_endpoint=None):
        # Load environment variables if necessary
        load_dotenv("/content/KEY.env")

        if not openai_api_key:
            openai_api_key = os.getenv("OPENAI_API_KEY")
            if not openai_api_key:
                raise ValueError("OpenAI API key is required. Please provide it or set it in the .env file.")

        if not azure_endpoint:
            azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
            if not azure_endpoint:
                raise ValueError("Azure OpenAI endpoint is required. Please provide it or set it in the .env file.")

        self.model_name = model_name
        self.azure_chat_model = AzureChatOpenAI(
            openai_api_key=openai_api_key,
            model = self.model_name,
            openai_api_version="2024-08-01-preview"
        )

    def generate_prompt(self, structure, text):
        system_message = (
            f"You are an expert at extracting structured data from text. \n"
            f"Your task is to convert the provided car description into a JSON format that matches the following structure: {structure}. "
            f"Ensure all values in the output JSON strictly adhere to the specified types in the structure. \n"
            f"Determine whether the price mentioned in the text is a direct price or an estimated price. Use the field name 'price' if it is explicitly stated as a price, or 'estimated_price' if it is described as an estimate. Convert the price to an integer amount in the smallest currency unit (e.g., 220K L.E. becomes 220000). \n"
            f"Pay close attention to details like numbers, strings, and nested structures. \n"
            f"Here is the text to process:\n\n{text}\n\n"
            f"Provide only the JSON output as a string without any additional explanation or commentary. "
        )
        return [("system", system_message), ("human", text)]

    def replace_body_type(self, llm_output, predicted_body_type):
        pattern = r'"body_type":\s*"(.*?)"'
        updated_json_string = re.sub(pattern, f'"body_type": "{predicted_body_type}"', llm_output)
        return updated_json_string

    def process_car_info(self, input_text, car_image, json_structure=None):
        # Default structure if not provided
        if json_structure is None:
            json_structure = {
                "car": {
                    "body_type": "Unknown",
                    "color": "string",
                    "brand": "string",
                    "model": "string",
                    "manufactured_year": "integer",
                    "motor_size_cc": "integer",
                    "tires": {
                        "type": "string",
                        "manufactured_year": "integer"
                    },
                    "windows": "string",
                    "notices": [
                        {
                            "type": "string",
                            "description": "string"
                        }
                    ],
                    "price_or_estimated_price": {
                        "amount": "integer",
                        "currency": "string"
                    }
                }
            }

        prompt = self.generate_prompt(json_structure, input_text)
        try:

            response = self.azure_chat_model.invoke(prompt)
            model_output = response.content
            predicted_body_type = CarTypePredictor().predict(car_image)
            updated_json_string = self.replace_body_type(model_output, predicted_body_type)

            return updated_json_string, predicted_body_type

        except OutputParserException as e:
            print("Error parsing the model's output:", e)
            return {}
        except Exception as e:
            print("An error occurred:", e)
            return {}


# Send Email

In [6]:
class EmailSender:
    def __init__(self, sender_email, sender_password, smtp_server=None, smtp_port=None):
        """
        Initializes the EmailSender with sender email and password, and optionally custom SMTP server details.
        """
        self.sender_email = sender_email
        self.sender_password = sender_password
        self.smtp_server = smtp_server
        self.smtp_port = smtp_port

    def _get_smtp_details(self):
        """
        Determines SMTP server and port based on the sender's email domain.
        """
        if "@outlook.com" in self.sender_email or "@hotmail.com" in self.sender_email:
            return "smtp.office365.com", 587
        elif "@gmail.com" in self.sender_email:
            return "smtp.gmail.com", 587
        elif "@yahoo.com" in self.sender_email:
            return "smtp.mail.yahoo.com", 465  # Yahoo typically uses SSL
        else:
            raise ValueError("Unsupported email provider. Please use Outlook, Gmail, Hotmail, or Yahoo.")

    def send_email(self, subject, body, recipient_email, image_path=None):
        """
        Sends an email with both text and optional image attachment.
        """
        try:
            # If custom SMTP details are not provided, get them based on sender's email
            smtp_server, smtp_port = self.smtp_server or self._get_smtp_details()

            # Create the message object
            message = MIMEMultipart()
            message["From"] = self.sender_email
            message["To"] = recipient_email
            message["Subject"] = subject

            # Attach the body text
            message.attach(MIMEText(body, "plain"))

            # Attach the image (if provided)
            if image_path and os.path.exists(image_path):
                with open(image_path, "rb") as image_file:
                    img = MIMEImage(image_file.read())
                    img.add_header("Content-Disposition", f"attachment; filename={os.path.basename(image_path)}")
                    message.attach(img)

            # Send the email
            if smtp_port == 465:
                with smtplib.SMTP_SSL(smtp_server, smtp_port) as server:
                    server.login(self.sender_email, self.sender_password)
                    server.sendmail(self.sender_email, recipient_email, message.as_string())
            else:  # Use TLS for Outlook and Gmail
                with smtplib.SMTP(smtp_server, smtp_port) as server:
                    server.starttls()  # Secure the connection
                    server.login(self.sender_email, self.sender_password)
                    server.sendmail(self.sender_email, recipient_email, message.as_string())

            print("Email sent successfully!")
            return True
        except Exception as e:
            print(f"Failed to send email: {e}")
            return False


# MAIN

In [9]:
def on_submit(car_description, sender_email, sender_pass, reciever_email, car_image):
    start_time = time.time()

    processor = CarDataProcessor()
    _ = processor.process_inputs(car_description, sender_email, sender_pass, reciever_email, car_image)
    email_sender = EmailSender(sender_email, sender_pass)

    subject = "The Car Image with the Generated JSON"
    body, predicted_car_type = CarInformationExtractor().process_car_info(input_text=car_description, car_image="")

    email_sent = email_sender.send_email(subject, body, reciever_email, image_path=processor.image_path)

    end_time = time.time()
    time_taken = round(end_time - start_time, 2)

    if email_sent:
        return predicted_car_type, f"Email sent successfully! Time taken: {time_taken} seconds"
    else:
        return predicted_car_type, "Failed to send email. Please check your details and try again."

In [10]:
demo = gr.Interface(
    fn=on_submit,
    inputs=[
        gr.Textbox(label="Car Description", lines=4),
        gr.Textbox(label="Sender Email"),
        gr.Textbox(label="Sender Email Password (Use App password for more security don't use your actual Password)"),
        gr.Textbox(label="Receiver Email"),
        gr.Image(label="Upload Car Image")
    ],
    outputs=[
        gr.Textbox(label="Predicted Car Type"),
        gr.Textbox(label="Time Taken")
    ]
)

demo.launch(debug = True)

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2c9707ba8e3de1d480.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Email sent successfully!
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://2c9707ba8e3de1d480.gradio.live
